# Подготовка

In [23]:
import os
# from pathlib import Path
# import shutil
import json
import requests
# from urllib.parse import quote
import re

# Константы
MYINDIE_JAM_URL = "https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page="
MYINDIE_JAM_URL_PAGES = 3  # Количество страниц с играми на геймджеме
OUTPUT_DIR = "output/"

In [20]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Скачать все страницы игр геймджема

Найти все игры на джеме

In [21]:
def get_jam_games_urls(jam_url):
    """Scrapes the jam page and extracts game URLs."""

    response = requests.get(jam_url)

    if response.status_code != 200:
        print(f"Error fetching jam page: `{response.status_code}`")
        return []

    games_urls = re.findall(r'/games/game/[\w-]+', response.text)
    games_urls = [f"https://myindie.ru{url}" for url in games_urls]
    print(f"Found {len(games_urls)} games URLs on: {jam_url}")

    return games_urls


games_urls = []
for page in range(1, MYINDIE_JAM_URL_PAGES + 1):
    paged_url = f"{MYINDIE_JAM_URL}{page}"
    games_urls.extend(get_jam_games_urls(paged_url))

print(f"\n{len(games_urls)} total games found across {MYINDIE_JAM_URL_PAGES} pages:\n{chr(10).join(games_urls)}")


Found 30 games URLs on: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=1
Found 30 games URLs on: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=2
Found 8 games URLs on: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=3

68 total games found across 3 pages:
https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
https://myindie.ru/games/game/trash-under-ground
https://myindie.ru/games/game/cult-indie
https://myindie.ru/games/game/udivitelnyj-ded
https://myindie.ru/games/game/franshiza-ktulhu_nx3
https://myindie.ru/games/game/durdom
https://myindie.ru/games/game/exorcismo
https://myindie.ru/games/game/5-days-with-the-necronomicon_opw
https://myindie.ru/games/game/full-moon-twin-rite
https://myindie.ru/games/game/zayachij-kult
https://myindie.ru/games/game/no-edward
https://myindie.ru/games/game/protokol-pentagrammy
https://myindie.ru/games/game/all-hail-sister
https://myindie.ru/games/game/s-run_5pm
https://myindie.ru/games/game/le

Скачать все HTML-страницы игр геймджема

In [22]:
jam_games_file_path = os.path.join(OUTPUT_DIR, f"jam_games.txt")
if os.path.exists(jam_games_file_path):
    os.remove(jam_games_file_path)
jam_games_file = open(jam_games_file_path, 'a', encoding='utf-8')

DEBUG_GAMES_MAX = 2  # DEBUG delete
for i, game_url in enumerate(games_urls[:DEBUG_GAMES_MAX]):
    print(f"Processing game URL: {game_url}")
    response = requests.get(game_url)
    if response.status_code != 200:
        print(f"* Error fetching game page: `{response.status_code}`")
        continue

    title_match = re.search(r'<title>([^<]+)</title>', response.text)
    if title_match:
        title = f"{i:03d}_" + re.sub(r'[^\w_]', '-', re.sub(r'\s+', '_', title_match.group(1).strip()))
    else:
        title = f"{i:03d}_" + "Unknown"

    output_file_path = os.path.join(OUTPUT_DIR, f"{title}.html")
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
        print(f"Saved to `{output_file_path}`")
    jam_games_file.write(f"{output_file_path} {game_url}\n")

jam_games_file.close()

Processing game URL: `https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom`
Saved to `output/000_УЛЬТИМАТУМ--_Одна_ночь_с_Карачуном-_Жанр-_Horror_-_Инди-игры_-_MyIndie.html`
Processing game URL: `https://myindie.ru/games/game/trash-under-ground`
Saved to `output/001_Fetidity-_Жанр-_-_Инди-игры_-_MyIndie.html`


# 2. Извлечь данные со всех страниц игр

In [53]:
def unflatten_nuxt_data(data):
    """ Разворачивает плоскую структуру данных Nuxt.js в дерево. """
    if not isinstance(data, list) or not data: return data
    memo = {}
    def resolve(val):
        if isinstance(val, int) and 0 <= val < len(data):
            if val not in memo:
                memo[val] = resolve_item(data[val])
            return memo[val]
        return val
    def resolve_item(item):
        if isinstance(item, dict):
            return {k: resolve(v) for k, v in item.items()}
        if isinstance(item, list):
            return [resolve(v) for v in item]
        return item
    return resolve_item(data[1])

jam_games_file_path = os.path.join(OUTPUT_DIR, "jam_games.txt")
jam_games_file = open(jam_games_file_path, 'r', encoding='utf-8')

all_judges_reviews = []

for line in jam_games_file:
    output_file_path, game_url = line.strip().split(' ', 1)
    print(f"Обработка: {game_url}")
    with open(output_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    match = re.search(r'id=\"__NUXT_DATA__\">([^<]+)</script>', html_content)
    if not match: continue

    json_data = json.loads(match.group(1))
    unflattened = unflatten_nuxt_data(json_data)

    # 1. Заходим в 'data'. Она содержит ['ShallowReactive', { ... }]
    data_section = unflattened.get('data', [])

    # 2. Берем второй элемент списка (индекс 1) — там словарь с результатами
    if isinstance(data_section, list) and len(data_section) > 1:
        payload_container = data_section[1]

        # 3. В словаре берем первый ключ (game<alias>)
        if isinstance(payload_container, dict) and payload_container:
            first_key = list(payload_container.keys())[0]
            game_payload = payload_container[first_key]

            # 4. Извлекаем отзывы из game_payload['data']['reviews']
            if isinstance(game_payload, dict):
                inner_data = game_payload.get('data', {})
                reviews = inner_data.get('reviews', [])

                # Собираем судей
                judges = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'judge']
                all_judges_reviews.extend(judges)
                print(f"  - Извлечено судейских отзывов: {len(judges)}")

jam_games_file.close()
print(f"\nВсего судейских отзывов: {len(all_judges_reviews)}")

Обработка: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
  - Извлечено судейских отзывов: 3

Всего судейских отзывов: 3
